# Day 9 — Incremental Data Ingestion with Auto Loader

### 1. Batch vs Incremental Ingestion

**Traditional Ingestion / BATCH ingestion** : 
    - you read every file again each day.
**Increamental Ingestion** :
    - you process ONLY New or changed data 

### 2. Auto Loader
- databricks feature to incrementally ingest the new file form cloud object storage and support Unity catalog volumes. 
- uses **_cloudFiles_** source with spark structured streaming.

````
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)
````

````
**Component	        Purpose**
readStream	        Creates a streaming DataFrame
cloudFiles	        Enables Auto Loader
cloudFiles.format	Specifies source format
schemaLocation	    Stores schema inference and evolution metadata
load()	            Specifies the input directory
````

**Auto Loader can run continuously or process available files and stop.**

### 3. Checkpoints
The checkpoint helps the streaming query recover its progress after a restart.

For example:
- First run: processes files 1, 2, and 3.
- Second run: file 4 arrives, so only file 4 needs processing.
- Third run: no new files, so no additional data is ingested.

### 4. Auto Loader versus COPY INTO
Both can ingest files incrementally, but they operate differently.
````
**Feature	            Auto Loader	                    COPY INTO**
Interface	        Structured Streaming	        SQL command
File discovery	    Incremental discovery	        Checks files against load history
Checkpoint	        Explicit streaming checkpoint	Managed load tracking
Schema evolution	Supported	                    More limited
Best suited for	    Scalable ongoing ingestion	    Simpler scheduled file loading
````

For a production pipeline receiving frequent files, Auto Loader is often a strong choice. For a relatively simple scheduled load, COPY INTO may be sufficient.

================================================================================================================================================================
Example--

1. create 2 dataframes and generate parquet files
2. Call Auto Loader to load s1 from parque fiel to customer360_bronze.customers_incremental_day09
3. add file- customers_03 parquet file to source directory
4. call Auto Loader to load the cust_03 data to customer360_bronze.customers_incremental_day09
5. add new column- emailId and add file- customers_04 parquet file to source directory
6. call Auto Loader with changes- add mergeSchema option to (1) Detects new columns in incoming files (2) update the defination. 

### Load 2 files using Auto Loader-->
````
from pyspark.sql import functions as F

# Create DataFrame with sample data
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("custId", IntegerType(), False),
    StructField("custName", StringType(), False),
    StructField("status", StringType(), False),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
])

# Separate volume for streaming metadata
spark.sql("""
CREATE VOLUME IF NOT EXISTS
workspace.customer360_bronze.ingestion_metadata
""")

spark.sql("""
CREATE VOLUME IF NOT EXISTS
workspace.customer360_bronze.raw_files
""")

# New source directory for the Parquet exercise
source_path = (
    "/Volumes/workspace/customer360_bronze/raw_files/parquet_source"
)

checkpoint_path = (
    "/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/customers"
)

schema_path = (
    "/Volumes/workspace/customer360_bronze/ingestion_metadata/schemas/customers"
)

# Create the initial source files
df1 = spark.createDataFrame(
    [
        (1001, "John Smith", "Active", "Dallas", "TX"),
        (1002, "Mary Jones", "Active", "Plano", "TX"),
        (1003, "Steve Brown", "Active", "Frisco", "TX")
    ],
    schema
)

df2 = spark.createDataFrame(
    [
        (1004, "Sarah Wilson", "Active", "Irving", "TX"),
        (1005, "David Lee", "Active", "McKinney", "TX")
    ],
    schema
)

df1.write.mode("overwrite").parquet(
    f"{source_path}/customers_01"
)

df2.write.mode("overwrite").parquet(
    f"{source_path}/customers_02"
)

# Configure Auto Loader
dfAutoLoader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)

# Ingest into Bronze
query = (
    dfAutoLoader.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(
        "workspace.customer360_bronze.customers_incremental_day09"
    )
)

query.awaitTermination()

# Verify
display(
    spark.table(
        "workspace.customer360_bronze.customers_incremental_day09"
    ).orderBy(F.col("custId"))
)

custId	custName	    status	city	    state	_rescued_data
1001	John Smith	    Active	Dallas	    TX	    null
1002	Mary Jones	    Active	Plano	    TX	    null
1003	Steve Brown	    Active	Frisco	    TX	    null
1004	Sarah Wilson	Active	Irving	    TX	    null
1005	David Lee	    Active	McKinney	TX	    null
````

### Add file -->
````
#Adding 3rd Parquet file to the source directory
df3 = spark.createDataFrame(
    [
        (1006, "Robert King", "Active", "Dallas", "TX"),
        (1007, "Jennifer White", "Active", "Plano", "TX")
    ],
    schema
)

df3.write.mode("overwrite").parquet(
    f"{source_path}/customers_03"
)

# Configure Auto Loader
dfAutoLoader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)

# Ingest into Bronze
query = (
    dfAutoLoader.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(
        "workspace.customer360_bronze.customers_incremental_day09"
    )
)

query.awaitTermination()

# Verify
display(
    spark.table(
        "workspace.customer360_bronze.customers_incremental_day09"
    ).orderBy(F.col("custId"))
)

custId	custName	    status	city	    state	_rescued_data
1001	John Smith	    Active	Dallas	    TX	    null
1002	Mary Jones	    Active	Plano	    TX	    null
1003	Steve Brown	    Active	Frisco	    TX	    null
1004	Sarah Wilson	Active	Irving	    TX	    null
1005	David Lee	    Active	McKinney	TX	    null
1006	Robert King	    Active	Dallas	    TX	    null
1007	Jennifer White	Active	Plano	    TX	    null
````

### Auto Loader Code --> MULTIPLE EXECUTION DOESNT ADD THE FILE AGAIN. NOTE-SCHEMAlOCATION MUST BE POINTING TO SAME PATH FOR TARGET FILE 
````
from pyspark.sql import functions as F

# Use the SAME source and metadata paths
source_path = ("/Volumes/workspace/customer360_bronze/raw_files/parquet_source")
target_table = ("workspace.customer360_bronze.customers_incremental_day09")

checkpoint_path = ( "/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/customers")
schema_path = ("/Volumes/workspace/customer360_bronze/ingestion_metadata/schemas/customers")


# Auto Loader with schema evolution
dfAutoLoader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

# Write to Delta with schema merging enabled
query = (
    dfAutoLoader.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

# Validate
display(
    spark.table(target_table)
    .orderBy("custId")
)

custId	custName	    status	city	    state	_rescued_data
1001	John Smith	    Active	Dallas	    TX	    null
1002	Mary Jones	    Active	Plano	    TX	    null
1003	Steve Brown	    Active	Frisco	    TX	    null
1004	Sarah Wilson	Active	Irving	    TX	    null
1005	David Lee	    Active	McKinney	TX	    null
1006	Robert King	    Active	Dallas	    TX	    null
1007	Jennifer White	Active	Plano	    TX	    null
````

**Why did we add mergeSchema?**
There are two separate schema evolution steps:
**Component	        Setting	                                            Purpose**
_Auto Loader source	cloudFiles.schemaEvolutionMode = addNewColumns	    Detects new columns in incoming files
Delta target	    mergeSchema = true	                                Allows the target table schema to include new columns_

Both matter. Auto Loader can recognize the new column, but the Delta target must also accept it.

**Interview-ready explanation**
In my Customer360 project, I implemented incremental ingestion using Databricks Auto Loader. When a new Parquet file arrived with an additional column, Auto Loader detected the schema change and stopped the stream. I restarted the pipeline using the existing checkpoint and schema location, and enabled schema merging in the Delta sink. This allowed the new column to be added without reprocessing the previously ingested files.


incremental ingestion, checkpoint recovery, and schema evolution

**Concept	Key                     takeaway**
Incremental ingestion	        Process newly arriving files instead of reprocessing all files
Auto Loader	                    Uses cloudFiles with Spark Structured Streaming
Checkpoint	                    Tracks processing progress and supports recovery
Schema location	                Stores schema inference and evolution metadata
availableNow=True	            Processes available data and then stops
Schema evolution	            Handles new columns arriving in source files
mergeSchema=true	            Allows the Delta target to accept additional columns
Bronze layer	                Preserves ingested data for downstream processing


1.** Initial ingestion — 5 records**
Created customers_01 and customers_02 Parquet datasets and loaded them into Bronze.

2.** Incremental ingestion — 7 records**
Added customers_03 containing two new customers. Auto Loader processed the new files.

3.** Checkpoint troubleshooting**
Discovered that Cell 3 used a different source and checkpoint, which caused duplicate ingestion. Corrected the paths and validated repeat runs.

4.** Schema evolution — 9 records**
Added customers_04 with a new emailId column. Auto Loader detected the schema change and stopped.

5. **Recovery and validation**
Reused the checkpoint, enabled Delta schema merging, restarted the stream, and successfully ingested 9 records. Subsequent runs kept the count at 9.


Remember the difference:
- **schemaLocation** — What does the incoming data look like?
- **checkpointLocation** — How far has this streaming query progressed?
- **mergeSchema** — Can the Delta target accept the new columns?

**Question: How have you implemented incremental ingestion using Databricks?**
- In my Customer360 practice project, I used Databricks Auto Loader to incrementally ingest Parquet files into a Bronze Delta table. 
- I configured a **checkpoint** to track processing progress and used the **AvailableNow** trigger for batch-style incremental execution.
- I also tested **schema evolution** by introducing a new column in an incoming file. 
- Auto Loader detected the schema change, and I restarted the pipeline using **the existing checkpoint** with schema merging enabled.
- I validated that only new files were processed and that repeated runs without new files didn't create duplicate records.